# Wikidata SPARQL notebook

Run SPARQL queries against the [Wikidata Query Service](https://query.wikidata.org/) and show results with Polars.

| | |
|---|---|
| **Endpoint** | `https://query.wikidata.org/sparql` |
| **UI** | https://query.wikidata.org/ |
| **Project** | Smart-grid knowledge queries |

Replace the sample query below with your own logic when ready.

In [ ]:
from __future__ import annotations

from typing import Any

import polars as pl
from SPARQLWrapper import JSON, SPARQLWrapper

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
USER_AGENT = "SmartGridNotebook/0.1 (https://github.com/local; educational)"


def run_sparql(query: str, endpoint: str = WIKIDATA_ENDPOINT) -> pl.DataFrame:
    """Execute a SPARQL SELECT query and return bindings as a Polars DataFrame."""
    client = SPARQLWrapper(endpoint)
    client.setQuery(query)
    client.setReturnFormat(JSON)
    client.addCustomHttpHeader("User-Agent", USER_AGENT)

    payload: dict[str, Any] = client.query().convert()
    bindings = payload.get("results", {}).get("bindings", [])
    rows = [{key: value.get("value") for key, value in row.items()} for row in bindings]
    return pl.DataFrame(rows) if rows else pl.DataFrame()

## Sample query

Example: a few power stations (instances of [Q159719](https://www.wikidata.org/wiki/Q159719)) with English labels.
Swap this query for your smart-grid use case later.

In [ ]:
SAMPLE_QUERY = """
SELECT ?item ?itemLabel ?countryLabel WHERE {
  ?item wdt:P31 wd:Q159719 .
  OPTIONAL { ?item wdt:P17 ?country . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 10
"""

df = run_sparql(SAMPLE_QUERY)
df